# Baseline — Deliverable 1
**Validación de inscripción de ramos desde una pregunta en prosa**

Punto de control de la Fase 4: correr un modelo de menos de 8B sobre los 21 casos piloto y mirar el acierto por nivel.

Ejecuta las celdas en orden. La corrida guarda caso a caso, así que si Colab se desconecta basta con volver a ejecutar la misma celda y retoma donde iba.

## 1. Confirmar la GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias
Tarda un par de minutos la primera vez.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Subir el paquete
Al ejecutar esta celda aparece un botón. Sube **`paquete_colab.zip`**.

In [ ]:
from google.colab import files
subidos = files.upload()
!unzip -o -q paquete_colab.zip -d proyecto
!ls -R proyecto

## 4. Verificar que el ground truth funciona acá también
Los 9 casos de control tienen que pasar antes de medir nada.

In [ ]:
%cd /content/proyecto/scripts
!python verificador.py | tail -3

## 5. Corrida A — zero-shot, pregunta en prosa
Es el baseline principal: la condición de *direct prompting* que pide el enunciado.

In [ ]:
!python runner.py --modelo Qwen/Qwen2.5-7B-Instruct --condicion zero_shot

## 6. Corrida B — zero-shot, pregunta limpia
**La corrida que hace el diagnóstico.** Mismos historiales y mismas respuestas correctas, pero con el código y el estado explícitos en vez de la prosa ambigua.

Si el acierto sube mucho respecto de la corrida A, queda probado que la falla es de interpretación. Si no cambia, la falla es de procedimiento. Cualquiera de los dos resultados es un hallazgo.

In [ ]:
!python runner.py --modelo Qwen/Qwen2.5-7B-Instruct --condicion zero_shot --limpia

## 7. Corrida C — few-shot, pregunta en prosa
Tres ejemplos resueltos delante. Cierra la objeción de *le preguntaste mal*: si el modelo sigue fallando con ejemplos, el problema no es el fraseo de la instrucción.

In [ ]:
!python runner.py --modelo Qwen/Qwen2.5-7B-Instruct --condicion few_shot

## 8. Comparación de las tres corridas

In [ ]:
import json, glob, os
print(f"{'corrida':46} {'n':>3} {'decisión':>10} {'regla':>9} {'conjunto':>10}")
print('-'*82)
for f in sorted(glob.glob('../resultados/*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r: continue
    p = lambda k: 100*sum(x[k] for x in r)/len(r)
    print(f"{os.path.basename(f)[:46]:46} {len(r):3} {p('acierto_decision'):9.1f}% "
          f"{p('acierto_regla'):8.1f}% {p('acierto_conjunto'):9.1f}%")
    for n in sorted({x['nivel'] for x in r}):
        s = [x for x in r if x['nivel']==n]
        q = lambda k: 100*sum(x[k] for x in s)/len(s)
        print(f"{'   nivel '+str(n):46} {len(s):3} {q('acierto_decision'):9.1f}% "
              f"{q('acierto_regla'):8.1f}% {q('acierto_conjunto'):9.1f}%")

## 9. Descargar los resultados
Bájalos y pásamelos: con eso analizo la taxonomía de errores y armo la figura del póster.

In [ ]:
!cd .. && zip -q -r resultados.zip resultados
from google.colab import files
files.download('/content/proyecto/resultados.zip')